# Custom contact map and weights

This tutorial covers an advanced usage of StructureDCA where:
- Instead of using the default distance-based contact map, the residue pairs for which couplings are inferred are set manually.
- Instead of using the default RSA-based `h`- and `J`-weights for StructureDCA[RSA], custom weights are set manually.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from structuredca import StructureDCA

### Example 1
### Step 1: Start StructureDCA but hold the DCA coefficient initialization

First, let us consider the previously used MBD domain.  
We will create a StructureDCA object and set the `init_dca` argument to `False`, in order to modify its contact map before inferring the sparse DCA `h` and `J` coefficients.

In [ ]:
# Create a StructureDCA object but hold the h and J coefficient initialization with: init_dca=False
sdca = StructureDCA(
    msa_path = '../test_data/6acv_A_29-94.fasta',
    pdb_path = '../test_data/6acv_A_29-94.pdb',
    chains = 'A',
    init_dca = False # this parameter prevents DCA initialization (computation of h and J coefficients)
)

# The structure has been aligned to the MSA target sequence
# The contact map was initialized using the default distance threshold of 8 angstroms
# Here, we plot the default StructureDCA contact map
plt.imshow(sdca.contacts, cmap='Greys')
plt.title("Default StructureDCA contact map")
plt.show()

### Example 1
### Step 2: Set a custom contact map and solve the DCA coefficients

For illustration, we exclude contacts between residues separated by fewer than `3` positions in the protein sequence.  
All other contacts are left unchanged.

In [ ]:
# Create a modified version of the contact map
# - NOTE: you can also modify sdca.contacts directly without copying it
WINDOW_SIZE = 3
L = sdca.msa_length
modified_contacts: np.ndarray = sdca.contacts.copy()
for i in range(L):
    for j in range(L):
        delta_window = abs(i - j)
        if 0 < delta_window < WINDOW_SIZE: # set to False residues separated by 1 or 2 positions in the sequence
            modified_contacts[i, j] = False

# Set the new contact map
sdca.contacts = modified_contacts

# Plot the new custom contact map
plt.imshow(sdca.contacts, cmap='Greys')
plt.title("Custom contact map (after applying the diagonal filter)")
plt.show()

# Now solve the DCA coefficients using the custom contact map to define the sparsity of the J coefficients
# (this step was previously held back by the argument init_dca=False)
sdca.init_dca()

### Note about custom contact map sanity checks

The custom contact map should respect some consistency criteria:
- It must be symmetric.
- The diagonal must always be `False` (no self-contacts).

Here is an illustration of an incorrect usage:

In [ ]:
# Create a StructureDCA object but hold the h and J coefficient initialization with: init_dca=False
sdca_wrong = StructureDCA(
    msa_path = '../test_data/6acv_A_29-94.fasta',
    pdb_path = '../test_data/6acv_A_29-94.pdb',
    chains = 'A',
    init_dca = False
)

# Modify the contacts in an inconsistent way: the matrix will not be symmetric
sdca_wrong.contacts[10, 11] = True
sdca_wrong.contacts[11, 10] = False

# ERROR because the contact map is inconsistent
try:
    sdca_wrong.init_dca()
except ValueError as err:
    print("\nERROR: ")
    print(err)

### Example 2
### Step 1: Derive a StructureDCA model based on two different 3D structures

Here, we consider a more realistic and complex example of StructureDCA with a custom contact map and custom `h`- and `J`-weights (for StructureDCA[RSA]).  
We aim to construct a StructureDCA model for the B1 β-lactamase family.  

Following what was done in [Chen et al.](https://www.nature.com/articles/s41467-024-52614-w), we investigate the epistasis between two proteins from this family: NDM1 and VIM2.  
Specifically, we study the effect of the background sequence on mutations in the two target proteins using the exact same StructureDCA model.  

We need to derive a single StructureDCA model that is equally suitable for both proteins.  
For this purpose, we build a StructureDCA model based on the same protein family MSA (`B1-beta-lactamases_msa.fasta`) and on both the structure of NDM1 (`3spu.pdb`) and VIM2 (`5yd7.pdb`):
- We consider a contact if the two residues are in contact in at least one of the two structures.
- We set the RSA-based weights as the average of the RSA-based weights derived from the two different structures.

**NOTE**: Despite the low sequence identity between NDM1 and VIM2, they are structurally and functionally similar.

In [ ]:
# First, we create two StructureDCA objects: one for NDM1 and one for VIM2
# We do not use these StructureDCA objects to perform DCA,
# but rather to compute the two contact maps (aligned to the target sequence) and the RSA-based weights
#  - NOTE: target sequences of NDM1 and VIM2 are trimmed and aligned to the protein family MSA,
#          and these alignments do contains some gaps.
#          However to perform the alignment to their corresponding 3D structures,
#          we use the same sequences but ungapped to better match the structure
sdca_ndm1 = StructureDCA(
    msa_path="../test_data/NDM1_trimmed_ungapped.fasta", # here we provide the FASTA file of the NDM1 sequence to align the structure
    pdb_path="../test_data/3spu.pdb", # NDM1 experimental 3D structure
    chains="A", # corresponding chain in the 3D structure
    init_dca=False, # prevent DCA coefficient initialization
)
sdca_vim2 = StructureDCA(
    msa_path="../test_data/VIM2_trimmed_ungapped.fasta", # here we provide the FASTA file of the VIM2 sequence to align the structure
    pdb_path="../test_data/5yd7.pdb", # VIM2 experimental 3D structure
    chains="A", # corresponding chain in the 3D structure
    init_dca=False, # prevent DCA coefficient initialization
)

# Now, we create a StructureDCA model with no structure (so it will be fully connected)
# We prevent DCA coefficient initialization (init_dca=False) in order to set custom contact maps and weights afterward
#  - NOTE: as NDM1 and VIM2 sequences aligned to the MSA do contain some gaps,
#          we have to consider gaps in the DCA model, so set exclude_gaps=False
sdca_beta_lactamases = StructureDCA(
    msa_path="../test_data/B1-beta-lactamases_msa.fasta", # protein family MSA
    pdb_path=None, # no reference structure (so DCA is fully connected by default)
    min_seqid=None, # no filtering on MSA sequences (use the MSA as provided)
    exclude_gaps=False, # do not exclude gap character from the DCA model and consider it a the 21th Amino Acid
    init_dca=False, # hold init_dca; we will first set custom contacts and RSA weights
)

# Set custom contact map:
# We consider a contact if there is at least one contact in NDM1 or in VIM2
L = sdca_beta_lactamases.msa_length
for i in range(L):
    for j in range(L):
        sdca_beta_lactamases.contacts[i, j] = sdca_ndm1.contacts[i, j] or sdca_vim2.contacts[i, j]

# Set custom RSA-based h- and J-weights (used for the StructureDCA[RSA] model):
# We define them as the average of the weights from NDM1 and VIM2
sdca_beta_lactamases.dca_model.wh = (sdca_ndm1.dca_model.wh + sdca_vim2.dca_model.wh) / 2.0 # shape=(L)
sdca_beta_lactamases.dca_model.wJ = (sdca_ndm1.dca_model.wJ + sdca_vim2.dca_model.wJ) / 2.0 # shape=(L, L)

# Solve the StructureDCA model with the custom contact map and weights
sdca_beta_lactamases.init_dca()

### Example 2
### Step 2: Observe epistasis between NDM1 and VIM2

Here, we illustrate the effect of the background sequence (epistasis) between NDM1 and VIM2 using the exact same StructureDCA model (derived previously).  
As observed, the two proteins display a relatively different mutational landscape despite being evaluated with the same model.  

This will require to use the `background_sequence` argument when evaluating effects of mutations.

In [ ]:
# Read NDM1 and VIM2 sequences (trimmed and aligned to match the protein family MSA)
#  - NOTE: these could also be hard coded as strings
#  - NOTE: NDM1 and VIM2 sequences aligned to the MSA do contain some gaps,
#          this is not a problem as we have set parameter exclude_gaps=False
from structuredca.sequence import FastaReader
ndm1_sequence: str = FastaReader.read_first_sequence("../test_data/NDM1_trimmed.fasta").sequence
vim2_sequence: str = FastaReader.read_first_sequence("../test_data/VIM2_trimmed.fasta").sequence

# Compute the effects of all possible single-site mutations (DMS) for both NDM1 and VIM2
# By default, StructureDCA uses the first sequence of the input MSA as the background sequence
# To override this behavior, we explicitly provide the `background_sequence` argument
dms_ndm1: list[dict] = sdca_beta_lactamases.eval_mutations_table(background_sequence=ndm1_sequence)
dms_vim2: list[dict] = sdca_beta_lactamases.eval_mutations_table(background_sequence=vim2_sequence)

# Compare mutational effects (ΔE) between NDM1 and VIM2
# We restrict the comparison to mutations occurring at positions
# where the wild-type amino acid is identical in both proteins
dE_ndm1, dE_vim2 = [], []
for mutation_ndm1, mutation_vim2 in zip(dms_ndm1, dms_vim2):
    if mutation_ndm1['mutation_fasta'] == mutation_vim2['mutation_fasta']: # check if same wild-type amino acid
        dE_ndm1.append(mutation_ndm1["StructureDCA"])
        dE_vim2.append(mutation_vim2["StructureDCA"])
plt.scatter(dE_ndm1, dE_vim2, label=f"Pearson: {np.corrcoef(dE_ndm1, dE_vim2)[0, 1]:.2f}")
plt.xlabel("$\\Delta E$ NDM1")
plt.ylabel("$\\Delta E$ VIM2")
plt.legend()
plt.show()

### Example 2
### Step 3: Background-specific sequence reconstruction

We have shown that the background sequence has an impact on StructureDCA predictions.  
We now test whether this effect meaningfully distinguishes NDM1 from VIM2.

For each background (NDM1 or VIM2), we reconstruct a predicted wild-type sequence by selecting at every position, the amino acid with the lowest ΔE (i.e., the most favorable mutation, including wild-type, given the background).  

We then compare how well each reconstructed sequence matches the true NDM1 and VIM2 sequences.  
The model properly captures background-dependent effects. Indeed, the reconstruction match the real protein sequence substantially better when the correct background is used.

In [ ]:
# Selects the most probable amino acid (lowest ΔE, highest P)
# at a given position for a specified background sequence
def select_most_probable_amino_acid(sdca: StructureDCA, i: int, background_sequence: str) -> str:
    amino_acid_probabilities = sdca.position_probabilities(i, background_sequence=background_sequence)
    best_amino_acid = None
    best_probability = 0.0
    for amino_acid, probability in amino_acid_probabilities.items():
        if probability > best_probability:
            best_probability = probability
            best_amino_acid = amino_acid
    return best_amino_acid

# Reconstruct the sequence one position at a time by choosing the most probable amino acid (lowest ΔE)
# (gapped positions are not reconstructed, and kept as gaps)
def reconstruct_sequence(sdca: StructureDCA, background_sequence: str) -> str:
    GAP = "-"
    seq: list[str] = []
    for i, wt_aa in enumerate(background_sequence):
        if wt_aa == GAP: # do not reconstruct gaps
            seq.append(GAP)
            continue
        aa = select_most_probable_amino_acid(sdca, i, background_sequence)
        seq.append(aa)
    return "".join(seq)

# Counte the number of matching positions between 2 sequences
# (gapped positions are ignoed)
def count_matching_positions(sequence1: str, sequence2: str) -> int:
    GAP = "-"
    n_match = 0
    for aa1, aa2 in zip(sequence1, sequence2):
        if aa1 == GAP or aa2 == GAP:
            continue
        if aa1 == aa2:
            n_match += 1
    return n_match

# Reconstruct the most probable sequence using each background
predicted_sequence_ndm1_background = reconstruct_sequence(sdca_beta_lactamases, ndm1_sequence)
predicted_sequence_vim2_background = reconstruct_sequence(sdca_beta_lactamases, vim2_sequence)

# Compare reconstructed sequences to the true sequences
# We expect higher agreement when the correct background is used
GAP = "-"
n_ungapped_positions = len([
    1 for aa1, aa2 in zip(ndm1_sequence, vim2_sequence)
    if aa1 != GAP or aa2 != GAP
])
ndm1_vs_ndm1_background_match = count_matching_positions(ndm1_sequence, predicted_sequence_ndm1_background)
print(f"NDM1 vs. reconstruction using NDM1 background: {ndm1_vs_ndm1_background_match} / {n_ungapped_positions} matches")

ndm1_vs_vim2_background_match = count_matching_positions(ndm1_sequence, predicted_sequence_vim2_background)
print(f"NDM1 vs. reconstruction using VIM2 background: {ndm1_vs_vim2_background_match} / {n_ungapped_positions} matches")

vim2_vs_ndm1_background_match = count_matching_positions(vim2_sequence, predicted_sequence_ndm1_background)
print(f"VIM2 vs. reconstruction using NDM1 background: {vim2_vs_ndm1_background_match} / {n_ungapped_positions} matches")

vim2_vs_vim2_background_match = count_matching_positions(vim2_sequence, predicted_sequence_vim2_background)
print(f"VIM2 vs. reconstruction using VIM2 background: {vim2_vs_vim2_background_match} / {n_ungapped_positions} matches")